# Fase 6 — Provincial Regression Modeling (Rebuilt)
## SC Tenerife + Las Palmas | Models P1, P2 + Lag Analysis (Las Palmas)

| Province | Islands | Temp proxy |
|---|---|---|
| SC Tenerife | TFE + La Palma + Gomera | Tenerife |
| Las Palmas | GC + Lanzarote + Fuerteventura | Gran Canaria |

**Period:** 2009–2025 | **Proxy:** v2 (AUC 0.886) | **SE:** HC3 robust

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

BASE = r"C:\Users\fdora\RA_Career\Projects\climate_mortality"

df_tfe = pd.read_parquet(f"{BASE}/data/processed/provinces/master_provincial_sc_tenerife_2009_2025.parquet")
df_lp  = pd.read_parquet(f"{BASE}/data/processed/provinces/master_provincial_las_palmas_2009_2025.parquet")

# First-difference vars
df_tfe = df_tfe.sort_values('week_start').reset_index(drop=True)
df_lp  = df_lp.sort_values('week_start').reset_index(drop=True)

df_tfe['deaths_diff'] = df_tfe['deaths'].diff()
df_lp['deaths_diff']  = df_lp['deaths'].diff()

# Calima lags (for Las Palmas analysis)
df_lp['calima_lag1'] = df_lp['calima_score_provincial'].shift(1)
df_lp['calima_lag2'] = df_lp['calima_score_provincial'].shift(2)

df_tfe.dropna(subset=['deaths_diff'], inplace=True)
df_lp.dropna(subset=['deaths_diff', 'calima_lag1', 'calima_lag2'], inplace=True)

print(f"SC Tenerife: {df_tfe.shape} | Las Palmas: {df_lp.shape}")
print(df_lp[['week_start','deaths','calima_score_provincial','calima_lag1','calima_lag2']].head())

SC Tenerife: (886, 8) | Las Palmas: (885, 10)
  week_start  deaths  calima_score_provincial  calima_lag1  calima_lag2
2 2009-01-19   151.0                 0.013283     0.172216     0.013283
3 2009-01-26   132.0                 0.344432     0.013283     0.172216
4 2009-02-02   160.0                 0.111717     0.344432     0.013283
5 2009-02-09   144.0                 0.030568     0.111717     0.344432
6 2009-02-16   139.0                 0.017285     0.030568     0.111717


In [2]:
print("SC Tenerife columns:", df_tfe.columns.tolist())
print("Las Palmas columns:", df_lp.columns.tolist())

SC Tenerife columns: ['week_start', 'deaths', 'deaths_missing', 'temp_c_mean', 'calima_score_provincial', 'calima_level_provincial', 'province', 'deaths_diff']
Las Palmas columns: ['week_start', 'deaths', 'deaths_missing', 'temp_c_mean', 'calima_score_provincial', 'calima_level_provincial', 'province', 'deaths_diff', 'calima_lag1', 'calima_lag2']


In [3]:
# Month (para estacionalidad)
df_tfe['month'] = pd.to_datetime(df_tfe['week_start']).dt.month
df_lp['month']  = pd.to_datetime(df_lp['week_start']).dt.month

# Lag deaths (para modelo P1)
df_tfe['lag_deaths'] = df_tfe['deaths'].shift(1)
df_lp['lag_deaths']  = df_lp['deaths'].shift(1)

df_tfe.dropna(subset=['lag_deaths'], inplace=True)
df_lp.dropna(subset=['lag_deaths'], inplace=True)

print(f"SC Tenerife: {df_tfe.shape} | Las Palmas: {df_lp.shape}")
print(df_lp[['week_start','deaths','lag_deaths','calima_score_provincial','calima_lag1','month']].head(3))

SC Tenerife: (885, 10) | Las Palmas: (884, 12)
  week_start  deaths  lag_deaths  calima_score_provincial  calima_lag1  month
3 2009-01-26   132.0       151.0                 0.344432     0.013283      1
4 2009-02-02   160.0       132.0                 0.111717     0.344432      2
5 2009-02-09   144.0       160.0                 0.030568     0.111717      2


## Model P2

In [6]:
formula_p2 = (
    "deaths_diff ~ calima_score_provincial "
    "+ C(month) "
    "+ temp_c_mean"
)

def fit_p2(df, label):
    model = smf.ols(formula=formula_p2, data=df).fit(cov_type='HC3')
    dw  = durbin_watson(model.resid)
    bp  = het_breuschpagan(model.resid, model.model.exog)
    _, sw = stats.shapiro(model.resid)
    b  = model.params['calima_score_provincial']
    p  = model.pvalues['calima_score_provincial']
    ci = model.conf_int().loc['calima_score_provincial']
    print(f"\n=== MODEL P2 HC3 — {label} ===")
    print(f"β calima:  {b:.3f}  |  p: {p:.3f}  |  95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"R²: {model.rsquared:.3f}  |  DW: {dw:.2f}  |  BP p: {bp[1]:.3f}  |  n: {int(model.nobs)}")
    return model, dw

m_p2_tfe, dw2_tfe = fit_p2(df_tfe, "SC Tenerife")
m_p2_lp,  dw2_lp  = fit_p2(df_lp,  "Las Palmas")


=== MODEL P2 HC3 — SC Tenerife ===
β calima:  7.519  |  p: 0.014  |  95% CI: [1.53, 13.50]
R²: 0.024  |  DW: 2.98  |  BP p: 0.000  |  n: 885

=== MODEL P2 HC3 — Las Palmas ===
β calima:  2.458  |  p: 0.436  |  95% CI: [-3.73, 8.65]
R²: 0.018  |  DW: 2.89  |  BP p: 0.030  |  n: 884


## Model  lag1 y lag2 for Las Palmas

In [7]:
# Lag 1: calima de la semana anterior
m_lag1 = smf.ols(
    "deaths_diff ~ calima_lag1 + C(month) + temp_c_mean",
    data=df_lp
).fit(cov_type='HC3')

# Lag 2: calima de hace dos semanas
m_lag2 = smf.ols(
    "deaths_diff ~ calima_lag2 + C(month) + temp_c_mean",
    data=df_lp
).fit(cov_type='HC3')

# Lag 1 + Lag 2 juntos (efecto acumulado)
m_lag12 = smf.ols(
    "deaths_diff ~ calima_score_provincial + calima_lag1 + calima_lag2 + C(month) + temp_c_mean",
    data=df_lp
).fit(cov_type='HC3')

for label, m, var in [
    ("Lag 0 (contemporáneo)", m_p2_lp,  'calima_score_provincial'),
    ("Lag 1 (1 semana)",      m_lag1,   'calima_lag1'),
    ("Lag 2 (2 semanas)",     m_lag2,   'calima_lag2'),
]:
    b  = m.params[var]
    p  = m.pvalues[var]
    ci = m.conf_int().loc[var]
    dw = durbin_watson(m.resid)
    print(f"{label:25s}  β={b:+.3f}  p={p:.3f}  CI=[{ci[0]:.2f},{ci[1]:.2f}]  DW={dw:.2f}")

print("\n--- Modelo combinado lag0+lag1+lag2 ---")
for var in ['calima_score_provincial','calima_lag1','calima_lag2']:
    b  = m_lag12.params[var]
    p  = m_lag12.pvalues[var]
    ci = m_lag12.conf_int().loc[var]
    print(f"  {var:30s}  β={b:+.3f}  p={p:.3f}  CI=[{ci[0]:.2f},{ci[1]:.2f}]")
print(f"  R²={m_lag12.rsquared:.3f}  DW={durbin_watson(m_lag12.resid):.2f}")

Lag 0 (contemporáneo)      β=+2.458  p=0.436  CI=[-3.73,8.65]  DW=2.89
Lag 1 (1 semana)           β=-1.433  p=0.624  CI=[-7.17,4.31]  DW=2.89
Lag 2 (2 semanas)          β=-2.222  p=0.464  CI=[-8.17,3.73]  DW=2.89

--- Modelo combinado lag0+lag1+lag2 ---
  calima_score_provincial         β=+3.161  p=0.333  CI=[-3.24,9.56]
  calima_lag1                     β=-1.586  p=0.590  CI=[-7.36,4.18]
  calima_lag2                     β=-2.117  p=0.481  CI=[-8.00,3.77]
  R²=0.019  DW=2.89


In [8]:
summary = pd.DataFrame([
    {'Model': 'P2 — Lag 0 (contemporaneous)', 'β': +2.458, 'p': 0.436, 'CI_low': -3.73, 'CI_high': 8.65, 'Significant': False},
    {'Model': 'P2 — Lag 1 (1 week)',          'β': -1.433, 'p': 0.624, 'CI_low': -7.17, 'CI_high': 4.31, 'Significant': False},
    {'Model': 'P2 — Lag 2 (2 weeks)',         'β': -2.222, 'p': 0.464, 'CI_low': -8.17, 'CI_high': 3.73, 'Significant': False},
    {'Model': 'P2 — Lag 0+1+2 combined',      'β': +3.161, 'p': 0.333, 'CI_low': -3.24, 'CI_high': 9.56, 'Significant': False},
])

print("=" * 70)
print("  LAS PALMAS — CALIMA LAG ANALYSIS (P2 HC3, first-difference OLS)")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("CONCLUSION:")
print("  No significant calima effect detected at any lag (0–2 weeks).")
print("  Lag coefficients are negative at lag1/lag2, inconsistent with")
print("  an inflammatory response mechanism.")
print("  Provincial aggregation (GC + Lanzarote + Fuerteventura) likely")
print("  dilutes the island-level signal observed in Gran Canaria alone")
print("  (β=+1.77, p<0.001 at island scale).")
print()
print("  Las Palmas provincial null result is robust across all specifications.")
print("  SC Tenerife reference: β=+7.519, p=0.014 ✅ (contemporaneous P2 HC3)")

  LAS PALMAS — CALIMA LAG ANALYSIS (P2 HC3, first-difference OLS)
                       Model      β     p  CI_low  CI_high  Significant
P2 — Lag 0 (contemporaneous)  2.458 0.436   -3.73     8.65        False
         P2 — Lag 1 (1 week) -1.433 0.624   -7.17     4.31        False
        P2 — Lag 2 (2 weeks) -2.222 0.464   -8.17     3.73        False
     P2 — Lag 0+1+2 combined  3.161 0.333   -3.24     9.56        False

CONCLUSION:
  No significant calima effect detected at any lag (0–2 weeks).
  Lag coefficients are negative at lag1/lag2, inconsistent with
  an inflammatory response mechanism.
  Provincial aggregation (GC + Lanzarote + Fuerteventura) likely
  dilutes the island-level signal observed in Gran Canaria alone
  (β=+1.77, p<0.001 at island scale).

  Las Palmas provincial null result is robust across all specifications.
  SC Tenerife reference: β=+7.519, p=0.014 ✅ (contemporaneous P2 HC3)
